# Maize Data Preprocessing

This notebook preprocesses `datasets/maize_data.csv` step by step. It creates two outputs:

- `datasets/maize_data_cleaned.csv`: cleaned readable data with an outlier flag.
- `datasets/maize_data_preprocessed.csv`: numeric, model-ready data with encoded categories and scaled features.

## Step 1: Import Libraries

In [1]:
from pathlib import Path

import pandas as pd

pd.set_option("display.max_columns", None)

## Step 2: Load the Dataset

In [2]:
DATA_PATH = Path("datasets/maize_data.csv")
CLEANED_PATH = Path("datasets/maize_data_cleaned.csv")
PREPROCESSED_PATH = Path("datasets/maize_data_preprocessed.csv")

df = pd.read_csv(DATA_PATH)
df.head()

,Crop,Crop_Year,Season,State,Area,Production,Annual_Rainfall,Fertilizer,Pesticide,Yield
0,Maize,1997,Kharif,Assam,19216.0,14721,2051.4,1828786.72,5956.96,0.615652
1,Maize,1997,Kharif,Karnataka,502797.0,1391132,1266.7,47851190.49,155867.07,2.687778
2,Maize,1997,Rabi,Karnataka,48844.0,98932,1266.7,4648483.48,15141.64,1.980000
3,Maize,1997,Summer,Karnataka,9730.0,20893,1266.7,926004.10,3016.30,2.165714
4,Maize,1997,Kharif,Meghalaya,17175.0,24878,3818.2,1634544.75,5324.25,1.444286


## Step 3: Basic Dataset Information

Check the number of rows, columns, data types, missing values, and duplicate records.

In [3]:
print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())
print("\nData types:")
print(df.dtypes)
print("\nMissing values:")
print(df.isna().sum())
print("\nDuplicate rows:", df.duplicated().sum())

Shape: (975, 10)

Columns:
['Crop', 'Crop_Year', 'Season', 'State', 'Area', 'Production', 'Annual_Rainfall', 'Fertilizer', 'Pesticide', 'Yield']

Data types:
Crop                object
Crop_Year            int64
Season              object
State               object
Area               float64
Production           int64
Annual_Rainfall    float64
Fertilizer         float64
Pesticide          float64
Yield              float64
dtype: object

Missing values:
Crop               0
Crop_Year          0
Season             0
State              0
Area               0
Production         0
Annual_Rainfall    0
Fertilizer         0
Pesticide          0
Yield              0
dtype: int64

Duplicate rows: 0


## Step 4: Clean Column Names and Text Values

The `Season` column contains extra spaces such as `Kharif     `. This step removes extra whitespace from column names and categorical values.

In [4]:
cleaned = df.copy()
cleaned.columns = cleaned.columns.str.strip()

text_columns = cleaned.select_dtypes(include="object").columns
for column in text_columns:
    cleaned[column] = cleaned[column].astype(str).str.strip()

cleaned[["Crop", "Season", "State"]].head()

,Crop,Season,State
0,Maize,Kharif,Assam
1,Maize,Kharif,Karnataka
2,Maize,Rabi,Karnataka
3,Maize,Summer,Karnataka
4,Maize,Kharif,Meghalaya


## Step 5: Convert Numeric Columns

Make sure all numerical columns are stored as numbers. Invalid numeric values, if any, are converted to missing values so they can be handled cleanly.

In [5]:
numeric_columns = [
    "Crop_Year",
    "Area",
    "Production",
    "Annual_Rainfall",
    "Fertilizer",
    "Pesticide",
    "Yield",
]

for column in numeric_columns:
    cleaned[column] = pd.to_numeric(cleaned[column], errors="coerce")

cleaned[numeric_columns].dtypes

Crop_Year            int64
Area               float64
Production           int64
Annual_Rainfall    float64
Fertilizer         float64
Pesticide          float64
Yield              float64
dtype: object

## Step 6: Handle Missing Values and Duplicates

Rows with missing values in required columns are removed. Duplicate rows are also removed.

In [6]:
required_columns = numeric_columns + ["Crop", "Season", "State"]

before_rows = len(cleaned)
cleaned = cleaned.drop_duplicates()
cleaned = cleaned.dropna(subset=required_columns)
cleaned = cleaned.sort_values(["Crop_Year", "State", "Season"]).reset_index(drop=True)

print("Rows before cleaning:", before_rows)
print("Rows after cleaning:", len(cleaned))
print("Remaining missing values:", cleaned.isna().sum().sum())
print("Remaining duplicate rows:", cleaned.duplicated().sum())

Rows before cleaning: 975
Rows after cleaning: 975
Remaining missing values: 0
Remaining duplicate rows: 0


## Step 7: Detect Yield Outliers

Yield outliers are identified with the IQR method. They are flagged in the cleaned dataset instead of being deleted immediately, so you can inspect them.

In [7]:
q1 = cleaned["Yield"].quantile(0.25)
q3 = cleaned["Yield"].quantile(0.75)
iqr = q3 - q1
lower_bound = q1 - 1.5 * iqr
upper_bound = q3 + 1.5 * iqr

cleaned["Yield_Outlier"] = ~cleaned["Yield"].between(lower_bound, upper_bound)

print("Lower bound:", lower_bound)
print("Upper bound:", upper_bound)
print("Yield outliers:", cleaned["Yield_Outlier"].sum())

cleaned.sort_values("Yield", ascending=False).head(10)

Lower bound: -0.48977678625000065
Upper bound: 4.73367559575
Yield outliers: 62


,Crop,Crop_Year,Season,State,Area,Production,Annual_Rainfall,Fertilizer,Pesticide,Yield,Yield_Outlier
14,Maize,1997,Autumn,Maharashtra,21.0,19695,1156.100000,1998.57,6.51,989.870000,True
583,Maize,2012,Kharif,Delhi,37.0,828,451.900000,5579.60,11.47,22.380000,True
631,Maize,2013,Kharif,Delhi,37.0,828,706.800000,5346.13,9.99,22.380000,True
679,Maize,2014,Kharif,Delhi,35.0,783,416.400000,5283.60,11.55,22.370000,True
537,Maize,2011,Kharif,Delhi,38.0,843,641.395455,6365.76,12.54,22.180000,True
450,Maize,2009,Kharif,Delhi,48.0,1054,528.400000,7479.36,8.16,21.960000,True
493,Maize,2010,Kharif,Delhi,40.0,878,989.500000,6644.40,9.60,21.950000,True
858,Maize,2017,Kharif,Tamil Nadu,166195.0,1532106,970.900000,26165740.80,63154.10,8.962000,True
959,Maize,2019,Kharif,Tamil Nadu,168587.0,1506526,910.100000,28956503.12,62377.19,8.614828,True
707,Maize,2014,Kharif,Tamil Nadu,191481.0,1663616,911.300000,28905971.76,63188.73,8.406538,True


## Step 8: Save the Cleaned Dataset

This output keeps readable category names and includes the `Yield_Outlier` flag.

In [8]:
cleaned.to_csv(CLEANED_PATH, index=False)
print("Saved:", CLEANED_PATH)
cleaned.head()

Saved: datasets\maize_data_cleaned.csv


,Crop,Crop_Year,Season,State,Area,Production,Annual_Rainfall,Fertilizer,Pesticide,Yield,Yield_Outlier
0,Maize,1997,Kharif,Andhra Pradesh,305100.0,791500,927.5,29036367.00,94581.00,2.201818,False
1,Maize,1997,Rabi,Andhra Pradesh,90800.0,291800,927.5,8641436.00,28148.00,3.210000,False
2,Maize,1997,Kharif,Arunachal Pradesh,34504.0,50025,2274.9,3283745.68,10696.24,1.686923,False
3,Maize,1997,Kharif,Assam,19216.0,14721,2051.4,1828786.72,5956.96,0.615652,False
4,Maize,1997,Kharif,Bihar,375000.0,456375,1303.7,35688750.00,116250.00,1.260185,False


## Step 9: Create Model-Ready Data

For modeling, this step removes yield outliers, drops the constant `Crop` column, and one-hot encodes categorical features.

In [9]:
model_data = cleaned.loc[~cleaned["Yield_Outlier"]].copy()

if model_data["Crop"].nunique() == 1:
    model_data = model_data.drop(columns=["Crop"])

model_data = model_data.drop(columns=["Yield_Outlier"])
model_data = pd.get_dummies(
    model_data,
    columns=["Season", "State"],
    prefix=["Season", "State"],
    dtype=int,
)

model_data.head()

,Crop_Year,Area,Production,Annual_Rainfall,Fertilizer,Pesticide,Yield,Season_Autumn,Season_Kharif,Season_Rabi,Season_Summer,Season_Whole Year,Season_Winter,State_Andhra Pradesh,State_Arunachal Pradesh,State_Assam,State_Bihar,State_Chhattisgarh,State_Delhi,State_Gujarat,State_Haryana,State_Himachal Pradesh,State_Jammu and Kashmir,State_Jharkhand,State_Karnataka,State_Kerala,State_Madhya Pradesh,State_Maharashtra,State_Manipur,State_Meghalaya,State_Mizoram,State_Nagaland,State_Odisha,State_Punjab,State_Sikkim,State_Tamil Nadu,State_Telangana,State_Tripura,State_Uttar Pradesh,State_Uttarakhand,State_West Bengal
0,1997,305100.0,791500,927.5,29036367.00,94581.00,2.201818,0,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
1,1997,90800.0,291800,927.5,8641436.00,28148.00,3.210000,0,0,1,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
2,1997,34504.0,50025,2274.9,3283745.68,10696.24,1.686923,0,1,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
3,1997,19216.0,14721,2051.4,1828786.72,5956.96,0.615652,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
4,1997,375000.0,456375,1303.7,35688750.00,116250.00,1.260185,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


## Step 10: Scale Numeric Feature Columns

The input features are standardized using z-score scaling. The target column `Yield` is not scaled.

In [10]:
feature_columns = [column for column in numeric_columns if column != "Yield"]

for column in feature_columns:
    mean = model_data[column].mean()
    std = model_data[column].std(ddof=0)
    if std != 0:
        model_data[column] = (model_data[column] - mean) / std

target = model_data.pop("Yield")
model_data["Yield"] = target

model_data.head()

,Crop_Year,Area,Production,Annual_Rainfall,Fertilizer,Pesticide,Season_Autumn,Season_Kharif,Season_Rabi,Season_Summer,Season_Whole Year,Season_Winter,State_Andhra Pradesh,State_Arunachal Pradesh,State_Assam,State_Bihar,State_Chhattisgarh,State_Delhi,State_Gujarat,State_Haryana,State_Himachal Pradesh,State_Jammu and Kashmir,State_Jharkhand,State_Karnataka,State_Kerala,State_Madhya Pradesh,State_Maharashtra,State_Manipur,State_Meghalaya,State_Mizoram,State_Nagaland,State_Odisha,State_Punjab,State_Sikkim,State_Tamil Nadu,State_Telangana,State_Tripura,State_Uttar Pradesh,State_Uttarakhand,State_West Bengal,Yield
0,-1.798747,0.556852,0.628117,-0.668697,0.187723,0.661790,0,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,2.201818
1,-1.798747,-0.276638,-0.129223,-0.668697,-0.364459,-0.213267,0,0,1,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,3.210000
2,-1.798747,-0.495594,-0.495654,1.092709,-0.509516,-0.443142,0,1,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1.686923
3,-1.798747,-0.555054,-0.549160,0.800535,-0.548908,-0.505568,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.615652
4,-1.798747,0.828719,0.120205,-0.176905,0.367833,0.947215,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1.260185


## Step 11: Save the Preprocessed Dataset

In [11]:
model_data.to_csv(PREPROCESSED_PATH, index=False)

print("Saved:", PREPROCESSED_PATH)
print("Cleaned shape:", cleaned.shape)
print("Preprocessed shape:", model_data.shape)
print("Missing values in preprocessed data:", model_data.isna().sum().sum())

Saved: datasets\maize_data_preprocessed.csv
Cleaned shape: (975, 11)
Preprocessed shape: (913, 41)
Missing values in preprocessed data: 0


## Step 12: Final Check

In [12]:
model_data.describe().T

,count,mean,std,min,25%,50%,75%,max
Crop_Year,913.0,1.189167e-14,1.000548,-1.798747,-0.882735,0.033276,0.796619,1.712630
Area,913.0,-1.211152e-16,1.000548,-0.629769,-0.588359,-0.449205,0.126988,5.266882
Production,913.0,-7.782505e-18,1.000548,-0.571471,-0.544498,-0.432255,0.142689,7.234503
Annual_Rainfall,913.0,-9.339006e-17,1.000548,-1.487305,-0.613269,-0.194291,0.353974,6.684916
Fertilizer,913.0,2.334752e-17,1.000548,-0.598406,-0.561637,-0.431876,0.105301,6.451928
Pesticide,913.0,9.339006e-17,1.000548,-0.584009,-0.550867,-0.417346,0.060955,6.804920
Season_Autumn,913.0,6.571742e-02,0.247923,0.000000,0.000000,0.000000,0.000000,1.000000
Season_Kharif,913.0,5.399781e-01,0.498672,0.000000,0.000000,1.000000,1.000000,1.000000
Season_Rabi,913.0,1.982475e-01,0.398898,0.000000,0.000000,0.000000,0.000000,1.000000
Season_Summer,913.0,1.555312e-01,0.362609,0.000000,0.000000,0.000000,0.000000,1.000000
